Script to Generate vector database from description and metadata
1. Create BaseModel: name, category,url,description,indicator_code

Need to use 
uv pip install chromadb
uv pip install litellm
uv pip install scikit-learn

In [ ]:
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from chromadb import PersistentClient
from tqdm import tqdm
from litellm import completion
import numpy as np
from sklearn.manifold import TSNE
import plotly.graph_objects as go
import pandas as pd
from pathlib import Path




load_dotenv(override=True)

MODEL = "gpt-4.1-nano"
DB_NAME = "intro/ind_vdb"
collection_name = "docs"
embedding_model = "text-embedding-3-large"
AVERAGE_CHUNK_SIZE = 500
indicators_input = "description"


openai = OpenAI()

In [ ]:
# Inspired by LangChain's Document - let's have something similar

class Result(BaseModel):
    page_content: str
    metadata: dict

In [ ]:
# A class to perfectly represent a chunk

class Chunk(BaseModel):
    headline: str = Field(description="A brief heading for this chunk, typically a few words, that is most likely to be surfaced in a query")
    original_text: str = Field(description="The original text of the chunk of the indicator")
    #url: str = Field(description="The url of indicator")
    #indicator_code: str = Field(description="The indicator code that used in reference to indicator")
    #name: str = Field(description="The name of the indicator")
    #category_code: str = Field(description="The dataset code that the indicator belongs to")
    #category_name: str = Field(description="The name of the dataset that the indicator belongs to")
    #category_url: str = Field(description="The url of the dataset that the indicator belongs to")
    
    def as_result( self, document):
        metadata = {"source": document["source"],
                    "category_code": document["category_code"],
                     "category_name": document["category_name"],
                     "category_url": document["category_url"],
                     "indicator_name": document["indicator_name"],
                     "indicator_url": document["indicator_url"],
                     "indicator_code": document["indicator_code"]}
        return Result(page_content=self.headline + "\n\n" + self.original_text,metadata=metadata)
    


In [ ]:
class Chunks(BaseModel):
    chunks: list[Chunk]

Fetch the description of indicators

load_indicators_csv() - Loads indicators from description/ind_list_with_descriptions.csv
load_datasets_csv() - Loads categories from description/datasets_list.csv
join_indicators_with_categories() - Joins the two DataFrames on category_id
create_documents_from_dataframe() - Creates document dictionaries with the required metadata
fetch_indicators() - Main function that orchestrates the above steps

In [ ]:


def load_indicators_csv() -> pd.DataFrame:
    """
    Load indicators from ind_list_with_descriptions.csv
    
    Returns:
        DataFrame with columns: id, name, url, category_id, description, indicator_code
    """
    csv_path = Path("description/ind_list_with_descriptions.csv")
    df = pd.read_csv(csv_path)
    return df


def load_datasets_csv() -> pd.DataFrame:
    """
    Load datasets/categories from datasets_list.csv
    
    Returns:
        DataFrame with columns: id, code, name, url, parent_id
    """
    csv_path = Path("description/datasets_list.csv")
    df = pd.read_csv(csv_path)
    return df


def join_indicators_with_categories(indicators_df: pd.DataFrame, datasets_df: pd.DataFrame) -> pd.DataFrame:
    """
    Join indicators with datasets/categories on category_id
    
    Args:
        indicators_df: DataFrame from load_indicators_csv()
        datasets_df: DataFrame from load_datasets_csv()
    
    Returns:
        Merged DataFrame with category information
    """
    # Rename columns in datasets_df to avoid conflicts
    datasets_renamed = datasets_df.rename(columns={
        'id': 'category_id',
        'code': 'category_code',
        'name': 'category_name',
        'url': 'category_url'
    })
    
    # Join on category_id
    merged_df = indicators_df.merge(
        datasets_renamed[['category_id', 'category_code', 'category_name']],
        on='category_id',
        how='left'
    )
    
    return merged_df


def create_documents_from_dataframe(df: pd.DataFrame) -> list[dict]:
    """
    Convert merged DataFrame into document dictionaries with metadata
    
    Args:
        df: Merged DataFrame from join_indicators_with_categories()
    
    Returns:
        List of document dictionaries with required fields
    """
    documents = []
    
    for _, row in df.iterrows():
        # Handle NaN values - fill with empty string or default values
        category_code = str(row.get('category_code', '')) if pd.notna(row.get('category_code')) else ''
        category_name = str(row.get('category_name', '')) if pd.notna(row.get('category_name')) else ''
        
        document = {
            "source": "ind_list_with_descriptions.csv",  # Fixed source
            "category_code": category_code,  # category_code from datasets
            "category_name": category_name,  # category_name from datasets
            "category_url": str(row.get('category_url', '')), # category_url from datasets
            "indicator_name": category_name,  # category_name from datasets
            "indicator_url": str(row.get('url', '')),
            "indicator_code": str(row.get('indicator_code', '')),
            "description": str(row.get('description', '')),
            "indicator_name": str(row.get('name', '')),
            "category_id": int(row.get('category_id', 0)) if pd.notna(row.get('category_id')) else 0
        }
        documents.append(document)
    
    return documents


def fetch_indicators() -> list[dict]:
    """
    Fetch the description of indicators from CSV files and join with category information.
    
    This function:
    1. Loads indicators from description/ind_list_with_descriptions.csv
    2. Loads categories from description/datasets_list.csv
    3. Joins them on category_id
    4. Creates document dictionaries with metadata fields
    
    Returns:
        List of document dictionaries, each containing:
        - source: "ind_list_with_descriptions.csv"
        - type: category_code from datasets
        - name: category_name from datasets
        - url: indicator URL
        - indicator_code: indicator code
        - description: indicator description
        - indicator_name: indicator name
        - category_id: category ID
    """
    # Load data
    indicators_df = load_indicators_csv()
    datasets_df = load_datasets_csv()
    
    # Join indicators with categories
    merged_df = join_indicators_with_categories(indicators_df, datasets_df)
    
    # Create documents
    documents = create_documents_from_dataframe(merged_df)
    
    return documents



In [ ]:
documents = fetch_indicators()


In [ ]:
print("number of indicators: " + str(len(documents)))

number of indicators: 2429


In [ ]:
documents[214]

{'source': 'ind_list_with_descriptions.csv',
 'category_code': 'EHEALTH_SURVEY',
 'category_name': 'EHEALTH_SURVEY: Global eHealth survey 2015',
 'category_url': '',
 'indicator_name': 'The importance of information sharing as a barrier to Big Data supporting universal health coverage',
 'indicator_url': 'https://gateway.euro.who.int/en/indicators/ehealth_survey_78-information-sharing-as-a-barrier-to-big-data/',
 'indicator_code': 'ehealth_survey_78',
 'description': 'This indicator measures the importance of information sharing as a barrier to big data supporting universal health coverage. Indicator code: ehealth_survey_78.',
 'category_id': 5}

STEP 2: Create Chunks. We skip this as we use each indicator as chunk

In [ ]:
# populate chunks  from documents
#[Chunk(**doc) for doc in documents]
def create_doc_as_chunks(documents: list[dict]) -> list[Chunk]:
    """
    Create Chunk objects from document dictionaries.
    
    Args:
        documents: List of document dictionaries from fetch_indicators()
    
    Returns:
        List of Result objects
    """
    docs_as_chunks = []
    
    for doc in documents:
        # Create a Chunk object from document fields
        chunk = Chunk(
            headline= "indicator:" + doc.get("indicator_name", "") + "; indicator category:" + doc.get("category_name", ""),
            original_text=" description:" + doc.get("description", "")  # Description is the original text
        )
        docs_as_chunks.append(chunk.as_result(doc))    
    return docs_as_chunks

In [ ]:
docs_as_chunks = create_doc_as_chunks(documents)

In [ ]:
len(docs_as_chunks)
docs_as_chunks[23]

Result(page_content='indicator:Travel distance to obtain assistive products for urban residents; indicator category:ASSISTIVETECH: ASSISTIVETECH\n\n description:This indicator measures travel distance to obtain assistive products for urban residents. Indicator code: at_23.', metadata={'source': 'ind_list_with_descriptions.csv', 'category_code': 'AT', 'category_name': 'ASSISTIVETECH: ASSISTIVETECH', 'category_url': '', 'indicator_name': 'Travel distance to obtain assistive products for urban residents', 'indicator_url': 'https://gateway.euro.who.int/en/indicators/at_23-travel-distance-to-obtain-assistive-products-for-urban-residents/', 'indicator_code': 'at_23'})

STEP3: store chunks in chromadb

In [20]:
def create_embeddings(chunks):
    #house keeping for chroma db
    chroma = PersistentClient(path=DB_NAME)
    if collection_name in [c.name for c in chroma.list_collections()]:
        chroma.delete_collection(collection_name)

    #send batch of 100 chunks to openai
    batch_size = 100
    all_embeddings = []
    for i in range(0, len(chunks), batch_size):
        batch_chunks = chunks[i:i+batch_size]
        batch_texts = [chunk.page_content for chunk in batch_chunks]
        
        # Create embeddings for this batch
        emb_response = openai.embeddings.create(
            model=embedding_model, 
            input=batch_texts
        )
        batch_embeddings = [e.embedding for e in emb_response.data]
        all_embeddings.extend(batch_embeddings)
        print(len(all_embeddings))
        
        # Optional: Add delay to respect rate limits
        #time.sleep(0.1)

    #emb = openai.embeddings.create(model=embedding_model, input=texts).data
    #vectors = [ e for e in all_embeddings]
    #getting all chunks page_content in one list
    texts = [chunk.page_content for chunk in chunks]
    collection = chroma.get_or_create_collection(collection_name)

    ids = [str(i) for i in range(len(chunks))]
    metas = [chunk.metadata for chunk in chunks]

    collection.add(ids=ids, embeddings=all_embeddings, documents=texts, metadatas=metas)
    print(f"Vectorstore created with {collection.count()} documents")

In [21]:

create_embeddings(docs_as_chunks)


100
200
300
400
500
600
700
800
900
1000
1100
1200
1300
1400
1500
1600
1700
1800
1900
2000
2100
2200
2300
2400
2429
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
Vectorstore created with 2429 documents


Visualize vectors


In [22]:
chroma = PersistentClient(path=DB_NAME)
collection = chroma.get_or_create_collection(collection_name)
result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
documents = result['documents']
metadatas = result['metadatas']
doc_types = [metadata['category_code'] for metadata in metadatas]
#colors = [['blue', 'green', 'red', 'orange'][['products', 'employees', 'contracts', 'company'].index(t)] for t in doc_types]
dataset_codes = [
    'H2020', 'HFA', 'ENHIS', 'HBSC', 'EHEALTH_SURVEY', 
    'CAH', 'HLTHRES', 'HFAMDB', 'INFL', 'MN_SURVEY', 
    'AMR', 'GNPR_SURVEY', 'CAHB_SURVEY', 'JMF', 'UHCFP', 
    'EPW', 'REHAB', 'HEPA', 'DH', 'AT'
]

color_palette = [
    'blue', 'green', 'red', 'orange', 'purple',
    'cyan', 'magenta', 'yellow', 'pink', 'brown',
    'gray', 'olive', 'navy', 'teal', 'coral',
    'lime', 'maroon', 'gold', 'indigo', 'salmon'
]

colors = [color_palette[dataset_codes.index(t)] for t in doc_types if t in dataset_codes]

In [ ]:
tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 2D scatter plot
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(title='2D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x',yaxis_title='y'),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()